# Multiple Comparisons

## Overview
Testing more than two groups without correction inflates the family-wise Type I error rate. With *k* tests each at α = 0.05, the probability of at least one false positive is 1 − 0.95^k — already 0.40 for 10 tests.

| Procedure | Controls | Use case |
|---|---|---|
| Tukey HSD | FWER | All pairwise, balanced ANOVA |
| Bonferroni | FWER | Any set; conservative |
| Holm | FWER | Sequential Bonferroni; always ≥ Bonferroni power |
| Dunnett | FWER | Each treatment vs one control |
| Benjamini-Hochberg | FDR | Many comparisons; exploratory |
| Planned contrasts | FWER (limited) | A priori hypotheses; most powerful |

**Quinn & Keough (2002) and Underwood (1997):** planned contrasts based on a priori hypotheses are more powerful and more scientifically defensible than post-hoc procedures applied after inspecting results.

---

In [ ]:
library(tidyverse); library(multcomp); library(emmeans)
set.seed(7)
# Intertidal disturbance experiment: 5 treatments × 12 replicates
treatments <- c("Control","Low","Medium","High","Removal")
richness <- data.frame(
  treatment = factor(rep(treatments, each=12), levels=treatments),
  richness  = c(rnorm(12,18,3), rnorm(12,16,3), rnorm(12,13,3),
                rnorm(12,10,3), rnorm(12, 8,3))
)
m <- aov(richness ~ treatment, data=richness)
cat("One-way ANOVA:\n"); print(summary(m))
cat("\nSignificant F → post-hoc comparisons appropriate\n")

---
## Tukey HSD and Dunnett's test

In [ ]:
# Tukey HSD — all pairwise
tukey <- TukeyHSD(m)
print(tukey)
# Compact letter display
library(multcompView)
print(multcompView::multcompLetters4(m, tukey))

cat("\n── Dunnett: each treatment vs Control only ──\n")
dunnett <- glht(m, linfct = mcp(treatment = "Dunnett"))
print(summary(dunnett))
cat("\nTukey makes", choose(5,2), "comparisons; Dunnett makes", 4,
    "→ Dunnett more powerful when only control comparisons matter\n")

---
## Bonferroni, Holm, and Benjamini-Hochberg

In [ ]:
methods <- c("none","bonferroni","holm","BH")
results <- lapply(methods, function(m_adj)
  pairwise.t.test(richness$richness, richness$treatment, p.adjust.method=m_adj))
sig_counts <- sapply(results, function(r) sum(r$p.value < 0.05, na.rm=TRUE))
names(sig_counts) <- c("Uncorrected","Bonferroni","Holm","BH (FDR)")
cat("Significant pairs at alpha=0.05:\n")
print(sig_counts)
cat("\nHolm is always >= as powerful as Bonferroni (use Holm by default).\n")
cat("BH controls FDR (expected false positives among significant results)\n")
cat("  rather than FWER — more power, accepts some false positives.\n")

---
## Planned (a priori) contrasts

In [ ]:
# Contrasts specified BEFORE seeing the data, from biological hypotheses
# Levels: Control  Low  Medium  High  Removal
K <- rbind(
  "Removal vs all others"  = c( 4, -1, -1, -1, -1) / 4,
  "Control vs disturbed"   = c( 3, -1, -1, -1,  0) / 3,
  "Linear disturbance grad"= c(-2, -1,  0,  1,  0)   # trend L→H
)
planned <- glht(m, linfct = mcp(treatment = K))
print(summary(planned))

# Verify orthogonality
cat("\nOrthogonality check (dot products of contrast pairs):\n")
cat("  C1 · C2:", sum(K[1,]*K[2,]), "\n")
cat("  C1 · C3:", sum(K[1,]*K[3,]), "\n")
cat("  C2 · C3:", sum(K[2,]*K[3,]), "\n")
cat("  Orthogonal contrasts provide independent (non-overlapping) information.\n")
cat("\nFewer comparisons → less severe correction → more power than all-pairwise\n")

---
## Common Pitfalls

**1. Running post-hoc tests after a non-significant omnibus F**
Post-hoc procedures generally require a significant overall F-test. Applying Tukey HSD to a non-significant ANOVA result and mining for significant pairs is not statistically valid. Planned contrasts with strong a priori justification are an exception.

**2. Conflating Holm and Bonferroni**
The Holm procedure is a uniformly more powerful sequential extension of Bonferroni. There is no situation in which plain Bonferroni is preferable to Holm. They are frequently confused in methods sections — always specify which was used.

**3. Claiming planned contrasts were planned when they were not**
A priori contrasts must be specified from the biological hypothesis before data are examined. If you look at the data, identify the most interesting pairs, then test those "as if" they were planned, you are inflating Type I error under the guise of planned comparisons.

**4. Using TukeyHSD on unbalanced designs**
Base R `TukeyHSD()` assumes equal group sizes. For unbalanced designs use `emmeans` with `pairs(..., adjust = "tukey")` which applies the Tukey-Kramer adjustment.

**5. Confusing FWER and FDR**
FWER = probability of any false positive. FDR = expected proportion of false positives among significant results. Tukey/Bonferroni/Holm control FWER; Benjamini-Hochberg controls FDR. The right choice depends on whether you can tolerate any false positives (confirmatory) or some (exploratory).

**6. Not reporting the correction method**
"Pairwise comparisons were conducted" is incomplete. Report the test, the correction method, and the family of comparisons it was applied to (all pairwise, vs control, etc.).


---
*r_methods_library - Samantha McGarrigle*